# Isaacus Reranker

This will help you get started with Isaacus reranking models using LangChain. For detailed documentation on `IsaacusRerank` features and configuration options, please refer to the [API reference](https://docs.langchain.com/oss/python/integrations/retrievers/isaacus_rerank).

## Overview
### Integration details

| Provider | Package |
|:--------:|:-------:|
| [Isaacus](/docs/integrations/providers/isaacus/) | [langchain-isaacus](https://docs.langchain.com/oss/python/integrations/retrievers/isaacus_rerank) |

## Setup

To access Isaacus reranking models you'll need to [create an Isaacus account](https://docs.isaacus.com/quickstart#1-set-up-your-account), [get an API key](https://docs.isaacus.com/quickstart#1-set-up-your-account), and install the `langchain-isaacus` integration package.

### Credentials

Head to the [Isaacus Platform](https://platform.isaacus.com/accounts/signup/) to sign up to Isaacus and generate an API key. Once you've done this set the ISAACUS_API_KEY environment variable to your API key value:

In [12]:
import getpass
import os

if not os.getenv("ISAACUS_API_KEY"):
    os.environ["ISAACUS_API_KEY"] = getpass.getpass("Enter your Isaacus API key: ")

To enable automated tracing of your model calls, set your [LangSmith](https://docs.smith.langchain.com/) API key:

In [13]:
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")

### Installation

The LangChain Isaacus integration lives in the `langchain-isaacus` package:

In [ ]:
%pip install -qU langchain-isaacus

## Instantiation

Now we can instantiate our model object and generate chat completions:

In [19]:
from langchain_isaacus import IsaacusEmbeddings, IsaacusRerank

rerank = IsaacusRerank(
    model="kanon-2-reranker", 
)

# we use Isaacus' embeddings for convenience in demonstration.
# You may use any embedding model you wish for the reranker.
embeddings = IsaacusEmbeddings(
    model="kanon-2-embedder",
)

## Indexing and Retrieval

Reranking models are often used in retrieval-augmented generation (RAG) flows, usually coming after the retrieval step. For more detailed instructions, please see our [RAG tutorials](/docs/tutorials/).

Below, see how to index and rerank retrieved documents using the `rerank` object we initialized above. In this example, we will index and retrieve a sample document in the `InMemoryVectorStore`.

In [ ]:
# Create a vector store with a sample text
# === if langchain installed === 
# import langchain
# langchain.debug = False, if langchain installed

from langchain_core.vectorstores import InMemoryVectorStore

query = "What is LangChain used for?"
texts = [
    "LangChain is a framework for building context-aware reasoning applications.",
    "It provides tools for retrieval-augmented generation (RAG) pipelines.",
    "Vector databases store embeddings for similarity search.",
    "Python is a popular programming language for AI and ML systems.",
    "Reranking improves retrieval quality by reordering candidate documents."
]

vectorstore = InMemoryVectorStore.from_texts(
    texts,
    embedding=embeddings,
) 

# Use the vectorstore as a retriever
retriever = vectorstore.as_retriever()

# Search for relevant texts
candidates = retriever.invoke(query)

# Rerank candidate texts using our reranker
reranked = rerank.compress_documents(
    query=query,
    documents=candidates
)

# Show the ranked results
print(query)
for idx, doc in enumerate(reranked):
    print(f"Rank {idx+1} ({doc.metadata["score"]:.4f}): {doc.page_content}")

What is LangChain used for?
Rank 1 (0.9427): LangChain is a framework for building context-aware reasoning applications.
Rank 2 (0.1717): It provides tools for retrieval-augmented generation (RAG) pipelines.
Rank 3 (0.0252): Vector databases store embeddings for similarity search.
Rank 4 (0.0223): Python is a popular programming language for AI and ML systems.


## Direct Usage

Under the hood, the vectorstore and retriever implementations are calling `embeddings.embed_documents(...)` and `embeddings.embed_query(...)` to create embeddings for the text(s) used in `from_texts` and retrieval `invoke` operations, respectively.

You can directly call these methods to get embeddings for your own use cases.

### Embed single texts

You can embed single texts or documents with `embed_query`:

## API Reference

For detailed documentation on `IsaacusRerank` features and configuration options, please refer to the [API reference](https://docs.langchain.com/oss/python/integrations/retrievers/isaacus_rerank).
